# Attention: If you run the notebook, please skip the “Train baseline CNN” cell, otherwise the baseline results may change due to randomness in training.

# Explainable Medical Image Triage Using Transfer Learning

##1. Data Loading

This section loads the PneumoniaMNIST dataset, visualizes sample images, and summarizes the class distribution.

In [ ]:
# Install packages
!pip -q install medmnist torch torchvision scikit-learn matplotlib

In [ ]:
# Import libraries

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from torchvision import transforms, models

import medmnist
from medmnist import INFO

# Reproducibility setup

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



In [ ]:
# Choose MedMNIST dataset

DATA_FLAG = "pneumoniamnist"

info = INFO[DATA_FLAG]
DataClass = getattr(medmnist, info["python_class"])

print("Dataset:", DATA_FLAG)
print("Task:", info["task"])
print("Number of classes:", len(info["label"]))
print("Labels:", info["label"])
print("Image channels:", info["n_channels"])

In [ ]:
# Define transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Load train / val / test datasets

train_dataset = DataClass(
    split="train",
    transform=train_transform,
    download=True,
    as_rgb=True
)

val_dataset = DataClass(
    split="val",
    transform=eval_transform,
    download=True,
    as_rgb=True
)

test_dataset = DataClass(
    split="test",
    transform=eval_transform,
    download=True,
    as_rgb=True
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

In [ ]:
# Create DataLoaders

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", labels[:10].squeeze().tolist())

In [ ]:
# Check class distribution

label_names = {int(k): v for k, v in info["label"].items()}

def show_class_distribution(dataset, split_name):
    labels = dataset.labels.squeeze()
    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{split_name} class distribution:")
    for cls, count in zip(unique, counts):
        print(f"Class {cls} ({label_names[int(cls)]}): {count}")

show_class_distribution(train_dataset, "Train")
show_class_distribution(val_dataset, "Validation")
show_class_distribution(test_dataset, "Test")

In [ ]:
# Class distribution table

def get_class_distribution(dataset, split_name):
    labels = dataset.labels.squeeze()
    unique, counts = np.unique(labels, return_counts=True)

    rows = []
    for cls, count in zip(unique, counts):
        rows.append({
            "Split": split_name,
            "Class ID": int(cls),
            "Class Name": label_names[int(cls)],
            "Count": int(count),
            "Percentage": round(count / len(dataset) * 100, 2)
        })
    return rows

distribution_rows = []
distribution_rows.extend(get_class_distribution(train_dataset, "Train"))
distribution_rows.extend(get_class_distribution(val_dataset, "Validation"))
distribution_rows.extend(get_class_distribution(test_dataset, "Test"))

class_distribution_df = pd.DataFrame(distribution_rows)
class_distribution_df

In [ ]:
# Plot class distribution

plt.figure(figsize=(8, 5))

for split_name, dataset in [
    ("Train", train_dataset),
    ("Validation", val_dataset),
    ("Test", test_dataset)
]:
    labels = dataset.labels.squeeze()
    unique, counts = np.unique(labels, return_counts=True)
    plt.bar(
        [f"{split_name}-{label_names[int(cls)]}" for cls in unique],
        counts
    )

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of Images")
plt.title("Class Distribution in PneumoniaMNIST")
plt.tight_layout()
plt.show()

PneumoniaMNIST is a binary medical image classification dataset with two classes: normal and pneumonia. The dataset is split into training, validation, and test sets. Since the class distribution is imbalanced, we report not only accuracy but also precision, recall, F1-score, false positives, and false negatives.

In [ ]:
# Display sample images

# Use a non-normalized transform for visualization.

viz_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

viz_dataset = DataClass(
    split="train",
    transform=viz_transform,
    download=True,
    as_rgb=True
)

viz_loader = DataLoader(viz_dataset, batch_size=12, shuffle=True)
viz_images, viz_labels = next(iter(viz_loader))

plt.figure(figsize=(12, 6))

for i in range(12):
    img = viz_images[i].permute(1, 2, 0).numpy()
    label_id = int(viz_labels[i].item())

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(label_names[label_id])
    plt.axis("off")

plt.suptitle("Sample Images from PneumoniaMNIST", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Baseline CNN

This section trains a baseline CNN from scratch. This baseline model will later be compared with transfer learning models such as ResNet18 or EfficientNet-B0

In [ ]:
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
# Baseline CNN model

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(BaselineCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


baseline_model = BaselineCNN(num_classes=2).to(device)
print(baseline_model)

In [ ]:
# Compute class weights for imbalanced data

train_labels = train_dataset.labels.squeeze()
unique_classes, class_counts = np.unique(train_labels, return_counts=True)

num_classes = len(unique_classes)
total_samples = len(train_labels)

class_weights = []
for cls in range(num_classes):
    count = class_counts[unique_classes.tolist().index(cls)]
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(baseline_model.parameters(), lr=1e-3)

In [ ]:
# Training and evaluation functions

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def evaluate_model(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.squeeze().long().to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)

    metrics = {
        "loss": epoch_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "confusion_matrix": confusion_matrix(all_labels, all_preds),
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs)
    }

    return metrics

# Please skip this “Train baseline CNN” cell! DO NOT RERUN!

In [ ]:
# Train baseline CNN

NUM_EPOCHS = 5

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_val_f1 = -1
best_model_state = None
best_epoch = None

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        baseline_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_metrics = evaluate_model(
        baseline_model,
        val_loader,
        criterion,
        device
    )

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])

    if val_metrics["f1"] > best_val_f1:
       best_val_f1 = val_metrics["f1"]
       best_model_state = copy.deepcopy(baseline_model.state_dict())
       best_epoch = epoch + 1


    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | Val F1: {val_metrics['f1']:.4f}"
    )

# Load best validation model
baseline_model.load_state_dict(best_model_state)
print(f"Best validation F1: {best_val_f1:.4f} at epoch {best_epoch}")

In [ ]:
# Convert training history to DataFrame

history_df = pd.DataFrame(history)
history_df

In [ ]:
# Evaluate the best baseline CNN on the test set

test_metrics = evaluate_model(
    baseline_model,
    test_loader,
    criterion,
    device
)

print("Baseline CNN Test Results")
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall / Sensitivity: {test_metrics['recall']:.4f}")
print(f"F1-score: {test_metrics['f1']:.4f}")

print("\nConfusion Matrix:")
print(test_metrics["confusion_matrix"])

print("\nClassification Report:")
print(classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=[label_names[0], label_names[1]],
    zero_division=0
))

In [ ]:
# Plot confusion matrix for baseline CNN

cm = test_metrics["confusion_matrix"]

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Baseline CNN Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(label_names))
plt.xticks(tick_marks, [label_names[i] for i in range(len(label_names))], rotation=45)
plt.yticks(tick_marks, [label_names[i] for i in range(len(label_names))])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Plot training and validation loss

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation accuracy

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Save baseline CNN results

baseline_results = pd.DataFrame([{
    "Model": "Baseline CNN",
    "Accuracy": test_metrics["accuracy"],
    "Precision": test_metrics["precision"],
    "Recall/Sensitivity": test_metrics["recall"],
    "F1-score": test_metrics["f1"],
    "False Positives": int(test_metrics["confusion_matrix"][0, 1]),
    "False Negatives": int(test_metrics["confusion_matrix"][1, 0])
}])

baseline_results.to_csv("baseline_cnn_results.csv", index=False)
baseline_results

In [ ]:
# Save baseline CNN model checkpoint

checkpoint = {
    "model_name": "BaselineCNN",
    "model_state_dict": baseline_model.state_dict(),
    "image_size": IMAGE_SIZE,
    "label_names": label_names,
    "test_metrics": {
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "confusion_matrix": test_metrics["confusion_matrix"].tolist()
    }
}

torch.save(checkpoint, "baseline_cnn_pneumoniamnist.pth")
print("Saved baseline model checkpoint: baseline_cnn_pneumoniamnist.pth")

The baseline CNN reached its strongest validation F1-score at epoch 5, while the lowest validation loss occurred at epoch 3. This suggests that validation performance fluctuated across epochs, but the model had largely converged within five epochs. Since the final model selection was based on validation F1-score, the epoch 5 checkpoint was selected for test evaluation.

On the test set, the baseline CNN achieved high recall/sensitivity but lower precision. This means the model missed very few pneumonia cases, which is useful for a triage setting, but it also produced many false positives, increasing the number of normal images flagged for review. Therefore, this baseline is useful for comparison, but the transfer learning model should aim to improve the precision-recall balance and reduce unnecessary false alerts.

# Transfer Learning Model

In [ ]:
# ============================================================
# 3. Transfer Learning Model: Fine-Tuned ResNet18
# ============================================================

# Load pretrained ResNet18
resnet18_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final layer for PneumoniaMNIST: normal vs pneumonia
num_features = resnet18_model.fc.in_features
resnet18_model.fc = nn.Linear(num_features, 2)

resnet18_model = resnet18_model.to(device)

# Fine-tuning: all layers are trainable
for param in resnet18_model.parameters():
    param.requires_grad = True

print(resnet18_model)

In [ ]:
# Loss and optimizer for fine-tuning

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Lower learning rate than baseline CNN because pretrained weights are being updated
optimizer = optim.Adam(resnet18_model.parameters(), lr=1e-5)

In [ ]:
# Train fine-tuned ResNet18

NUM_EPOCHS = 10

resnet_history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_val_f1 = -1
best_resnet_state = None
best_resnet_epoch = None

patience = 3
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        resnet18_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_metrics = evaluate_model(
        resnet18_model,
        val_loader,
        criterion,
        device
    )

    resnet_history["epoch"].append(epoch + 1)
    resnet_history["train_loss"].append(train_loss)
    resnet_history["train_accuracy"].append(train_acc)
    resnet_history["val_loss"].append(val_metrics["loss"])
    resnet_history["val_accuracy"].append(val_metrics["accuracy"])
    resnet_history["val_precision"].append(val_metrics["precision"])
    resnet_history["val_recall"].append(val_metrics["recall"])
    resnet_history["val_f1"].append(val_metrics["f1"])

    if val_metrics["f1"] > best_val_f1:

        best_val_f1 = val_metrics["f1"]

        best_resnet_state = copy.deepcopy(
            resnet18_model.state_dict()
        )

        best_resnet_epoch = epoch + 1

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping")
        break

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | Val F1: {val_metrics['f1']:.4f} | "
        f"Epochs Without Improvement: {epochs_without_improvement} | Patience: {patience}"
    )

    if epochs_without_improvement >= patience:
      print("Early stopping")
      break
# Load best validation model
resnet18_model.load_state_dict(best_resnet_state)
print(f"Best validation F1: {best_val_f1:.4f} at epoch {best_resnet_epoch}")

In [ ]:
# Convert ResNet18 training history to DataFrame

resnet_history_df = pd.DataFrame(resnet_history)
resnet_history_df

In [ ]:
# Evaluate the best fine-tuned ResNet18 on the test set

resnet_test_metrics = evaluate_model(
    resnet18_model,
    test_loader,
    criterion,
    device
)

print("Fine-Tuned ResNet18 Test Results")
print(f"Test Loss: {resnet_test_metrics['loss']:.4f}")
print(f"Accuracy: {resnet_test_metrics['accuracy']:.4f}")
print(f"Precision: {resnet_test_metrics['precision']:.4f}")
print(f"Recall / Sensitivity: {resnet_test_metrics['recall']:.4f}")
print(f"F1-score: {resnet_test_metrics['f1']:.4f}")

print("\nConfusion Matrix:")
print(resnet_test_metrics["confusion_matrix"])

print("\nClassification Report:")
print(classification_report(
    resnet_test_metrics["labels"],
    resnet_test_metrics["preds"],
    target_names=[label_names[0], label_names[1]],
    zero_division=0
))

In [ ]:
# Plot confusion matrix for fine-tuned ResNet18

cm = resnet_test_metrics["confusion_matrix"]

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Fine-Tuned ResNet18 Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(label_names))
plt.xticks(tick_marks, [label_names[i] for i in range(len(label_names))], rotation=45)
plt.yticks(tick_marks, [label_names[i] for i in range(len(label_names))])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Plot training and validation loss

plt.figure(figsize=(7, 5))
plt.plot(resnet_history_df["epoch"], resnet_history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(resnet_history_df["epoch"], resnet_history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-Tuned ResNet18 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation accuracy

plt.figure(figsize=(7, 5))
plt.plot(resnet_history_df["epoch"], resnet_history_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(resnet_history_df["epoch"], resnet_history_df["val_accuracy"], marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-Tuned ResNet18 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Save fine-tuned ResNet18 results

resnet_results = pd.DataFrame([{
    "Model": "Fine-Tuned ResNet18",
    "Accuracy": resnet_test_metrics["accuracy"],
    "Precision": resnet_test_metrics["precision"],
    "Recall/Sensitivity": resnet_test_metrics["recall"],
    "F1-score": resnet_test_metrics["f1"],
    "False Positives": int(resnet_test_metrics["confusion_matrix"][0, 1]),
    "False Negatives": int(resnet_test_metrics["confusion_matrix"][1, 0])
}])

resnet_results.to_csv("finetuned_resnet18_results.csv", index=False)
resnet_results

In [ ]:
# Save fine-tuned ResNet18 model checkpoint

checkpoint = {
    "model_name": "FineTunedResNet18",
    "model_state_dict": resnet18_model.state_dict(),
    "image_size": IMAGE_SIZE,
    "label_names": label_names,
    "test_metrics": {
        "accuracy": resnet_test_metrics["accuracy"],
        "precision": resnet_test_metrics["precision"],
        "recall": resnet_test_metrics["recall"],
        "f1": resnet_test_metrics["f1"],
        "confusion_matrix": resnet_test_metrics["confusion_matrix"].tolist()
    }
}

torch.save(checkpoint, "finetuned_resnet18_pneumoniamnist.pth")
print("Saved fine-tuned ResNet18 checkpoint: finetuned_resnet18_pneumoniamnist.pth")

In [ ]:
from google.colab import files

files.download("finetuned_resnet18_pneumoniamnist.pth")